In [7]:
import os, json, copy
from tqdm.notebook import tqdm
import sys

%load_ext autoreload
%autoreload 2
# Add a path to the beginning of sys.path (giving it priority)
sys.path.insert(0, "/mnt/ssd/workspace/adi/repos/vh_deepfake_trainer/tools")
#os.chdir("/mnt/ssd/workspace/adi/repos/vh_deepfake_trainer")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
from utils.json_utils import merge_nested_dicts,read_att_json

#### Naive split, use split indices

In [9]:
def split_indices_non_repeating(N, seed=None):
    #if seed is not None:
    #    random.seed(seed)
    
    indices = list(range(N))
    #random.shuffle(indices)

    train_end = int(0.5 * N)
    val_end = train_end + int(0.25 * N)

    train_indices = indices[:train_end]
    val_indices = indices[train_end:val_end]
    test_indices = indices[val_end:]

    return train_indices, val_indices, test_indices

#### Split by Attribute

##### Functions

##### Codes

In [ ]:
root_dir = "/mnt/ssd/workspace/adi/repos/vh_deepfake_trainer/tools/"
prefix = "200k_24nov_"
dataset_name = "200k_live_face_dataset"
path_json = os.path.join(root_dir,f"dataset_assessor/{prefix}{dataset_name}.json")
with open(path_json, 'r') as f:
    # Parsing the JSON file into a Python dictionary
    data_source = json.load(f)

if "200k_live_face_dataset" in path_json:
    data_source_new = {}
    for k, v in data_source.items():
        k_new = k.replace("/processes","")
        data_source_new[k_new] = v
    
    data_source = data_source_new

data_source_raw = data_source

In [11]:
dir_att = os.path.join(root_dir,f"dataset_assessor/results_att_fd3-1-0-24Nov/{dataset_name}")
dict_att_json = read_att_json(dir_att)
data_source = merge_nested_dicts(*[data_source_raw,dict_att_json],mode="existing_only")
data_source_2 = {k.split("/")[-1].split(".")[0]:v for k,v in data_source.items()}

In [12]:
dir_json = "/mnt/ssd/workspace/adi/repos/vh_deepfake_trainer/datasets/deepfake/dataset_json"
"""
list_json =  ["result_e4s_20251103-strictpose.json",
"result_blendface-strictpose.json",
"result_inswapper-strictpose.json",
"result_mobilefaceswap-strictpose.json",
"result_reswapper-strictpose.json"]
"""
list_json = ["200k_24nov_e4s.json",
"200k_24nov_blendface.json",
"200k_24nov_inswapper.json",
"200k_24nov_mobilefaceswap.json",
"200k_24nov_reswapper.json"]
list_dict_swap = []
for json_now in list_json:
    json_path = os.path.join(dir_json,json_now)

    with open(json_path, "r") as f:
        dict_swap = json.load(f)

    dict_swap_temp = {}
    for v in dict_swap.values():
        v_now = v
        v["swap_method"] = json_now.split(".")[0].split("_")[1]
        dict_swap_temp[v["image_path"]] = v
    
    dict_swap = dict_swap_temp
    list_dict_swap.append(dict_swap)

dict_swap = {k: v for d in list_dict_swap for k, v in d.items()}

for k,v in dict_swap.items():
    dict_swap[k]["source_name"] =  v["image_path"].split("/")[-1].split(".")[0].split("_to_")[0].replace('swap_','')
    dict_swap[k]["target_name"] =  v["image_path"].split("/")[-1].split(".")[0].split("_to_")[1].replace('swap_','')
#38368_verify_67dfb808-1e6c-47d6-b5ca-d23ae43dea20_11e03acf-3c0b-4e3d-96b1-6333e424a80f_original_to_169604_verify_90919c30-5031-4b92-a0cd-bb8e565bd660_4db71cc5-dbf6-4b36-9391-c00364c84101_original.jpg
#list_keys_1 = [i["image_path"].split("/")[-1].split("_to_")[0] for i in dict_att.values()]
#list_keys_2 = [i["image_path"].split("/")[-1].split("_to_")[1] for i in dict_att.values()]
#list_all.extend(list_keys_1+list_keys_2)
#list_all = list(set(list_all))

In [13]:
for k,v in dict_swap.items():
    data_info_source = data_source_2[v["source_name"]]
    data_info_target = data_source_2[v["target_name"]]
    dict_swap[k]["source_pred_age"]=data_info_source["pred_age"]
    dict_swap[k]["source_pred_gender"]=data_info_source["pred_gender"]
    dict_swap[k]["target_pred_age"]=data_info_target["pred_age"]
    dict_swap[k]["target_pred_gender"]=data_info_target["pred_gender"]

Split by Target Info

In [14]:
dict_swap[list(dict_swap.keys())[0]].keys()

dict_keys(['image_path', 'processed_path', 'label', 'method', 'dets', 'angle', 'head_pose', 'swap_method', 'source_name', 'target_name', 'source_pred_age', 'source_pred_gender', 'target_pred_age', 'target_pred_gender'])

In [15]:
# Create pandas DataFrame
list_image_key = []
dict_data_list = {}

for info_ctg in ["image_path","processed_path","source_name","target_name",
    "target_pred_age","target_pred_gender","swap_method"]:
    dict_data_list[info_ctg] = []

for k,v in dict_swap.items():
    list_image_key.append(k)
    for info_ctg in ["image_path","processed_path","source_name","target_name",
    "target_pred_age","target_pred_gender","swap_method"]:
        dict_data_list[info_ctg].append(v[info_ctg])


dict_data_list["swapped_id"] = list_image_key

In [16]:
import pandas as pd

#df_info = pd.DataFrame({"swapped_id":list_image_key,
#"source_name":list_source_name,"target_name":list_target_name,                        
#"target_pred_age":list_target_pred_age,"target_pred_gender":list_target_pred_gender})
df_info = pd.DataFrame(dict_data_list)
#df_info.groupby("target_pred_gender").count()
df_info

,image_path,processed_path,source_name,target_name,target_pred_age,target_pred_gender,swap_method,swapped_id
0,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,251891_verify_7e201ac0-8f9d-4154-95c1-2ee0b1ae...,62650_verify_8fe74056-3897-45a6-82f0-a5824f5c6...,40-49,FEMALE,24nov,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...
1,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,116386_verify_edfd7201-e3bc-4340-9902-c6c4c8b2...,45578_verify_212fbb8e-1d1c-4d6d-9034-bf8572b26...,10-19,MALE,24nov,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...
2,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,115209_verify_ddc8ccff-5eb3-422a-af62-236dcfa8...,1659_verify_4ca43735-010e-4b4f-9eba-ad527512d4...,10-19,MALE,24nov,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...
3,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,219429_verify_76d8172f-b04c-4aa6-be57-59c84031...,97001_verify_93a3e6c6-b9bb-41ed-909f-4f95b6984...,30-39,FEMALE,24nov,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...
4,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,216198_verify_1175ab28-2a38-4459-bab1-d70c29e0...,164613_verify_4f197d27-61f3-47a2-b775-a56441dc...,10-19,MALE,24nov,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...
...,...,...,...,...,...,...,...,...
578986,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,71146_verify_74d4afcb-0eb8-4ead-9205-4ece6e93f...,186126_verify_f9fa4de4-35b1-4e70-af3b-8bbbc6e3...,30-39,FEMALE,24nov,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...
578987,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,142085_compare_1_6113255e-fe62-4077-a6bb-a3478...,102025_verify_399b719e-31cd-4d30-a6c5-2b2cc49d...,3-9,MALE,24nov,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...
578988,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,242626_verify_edfe8730-9a11-4278-a2cb-62bcedf9...,245030_verify_8b6dd5ad-e068-4cb1-9a7e-df24752a...,20-29,MALE,24nov,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...
578989,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,151931_verify_9ce7d668-f3ca-4f75-8817-f93c5851...,224812_verify_9ff6ae26-7a36-461d-83ec-1ab02a77...,10-19,FEMALE,24nov,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...


Stratified split with Pandas DataFrame category

In [17]:
from sklearn.model_selection import train_test_split
import numpy as np
# 2. First split: Create training set and a temporary set (validation + test)
# This results in 60% train and 40% temp (0.6 train_size)
df_train, df_temp = train_test_split(df_info, train_size=0.6, stratify=df_info[['target_pred_gender',"swap_method"]]
                                     , random_state=42)

# 3. Second split: Divide the temporary set into validation and test sets
# This splits the 40% temp set 50/50, resulting in 20% validation and 20% test (0.5 test_size)
df_val, df_test = train_test_split(df_temp, test_size=0.5, stratify=df_temp[['target_pred_gender',"swap_method"]], random_state=42)

# 4. Verify the proportions and stratification
print(f"Train shape: {df_train.shape}")
print(f"Validation shape: {df_val.shape}")
print(f"Test shape: {df_test.shape}")

print("\nProportions in Train set:")
print(df_train['target_pred_gender'].value_counts(normalize=True).sort_index()) 
print(df_train['swap_method'].value_counts(normalize=True).sort_index()) 

Train shape: (347394, 8)
Validation shape: (115798, 8)
Test shape: (115799, 8)

Proportions in Train set:
target_pred_gender
FEMALE     0.606286
MALE       0.393700
UNKNOWN    0.000014
Name: proportion, dtype: float64
swap_method
24nov    1.0
Name: proportion, dtype: float64


In [18]:
df_train.head(5)

,image_path,processed_path,source_name,target_name,target_pred_age,target_pred_gender,swap_method,swapped_id
208276,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,89614_verify_c9893057-69ce-4385-bba1-4e06eda0e...,150493_verify_b9e4a880-e4a5-4efe-a603-63fb9dd2...,10-19,FEMALE,24nov,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...
61663,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,187566_verify_96653b16-3707-45dc-b700-103872b1...,227053_verify_4da81170-b682-4d0d-b2de-0953fc44...,40-49,FEMALE,24nov,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...
403932,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,181223_verify_2cf01681-947b-43b9-a4a9-367fc68e...,238852_verify_3cf27cea-bff8-4654-b5cf-d858fa22...,3-9,MALE,24nov,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...
321950,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,51275_verify_f45c7aba-63f0-4347-9778-643ea3ce6...,193755_verify_99d599fe-914a-4c3b-88c8-52101cdc...,40-49,MALE,24nov,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...
561196,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...,225275_verify_46445c75-05ab-435d-aacf-b11b7139...,111536_verify_cacd21ad-78e4-4d69-8820-d15406d7...,20-29,FEMALE,24nov,/mnt/ssd/workspace/adi/repos/vh_deepfake_train...


Recollect the real files information for each split

In [19]:
data_source_name_mapping = {v.split("/")[-1].split(".")[0]:v for v in list(data_source_raw.keys())}

In [20]:
list_real_name_train = list(df_train["source_name"]) + list(df_train["target_name"])
list_real_name_test = list(df_test["source_name"]) + list(df_test["target_name"])

In [21]:
dict_real_train = {data_source_name_mapping[i]:data_source_raw[data_source_name_mapping[i]] for i in list_real_name_train}
dict_real_test = {data_source_name_mapping[i]:data_source_raw[data_source_name_mapping[i]] for i in list_real_name_test}
print(f"{len(dict_real_train)}, {len(dict_real_test)}")

129818, 104160


In [22]:
output_json = os.path.join(dir_json,"test_200k_24nov_train.json")
with open(output_json, "w") as f:
    json.dump(dict_real_train, f, indent=4)

output_json = os.path.join(dir_json,"test_200k_24nov_test.json")
with open(output_json, "w") as f:
    json.dump(dict_real_test, f, indent=4)

In [23]:
dict_fake_train = {k:dict_swap[k] for k in list(df_train['image_path'])}
output_json = os.path.join(dir_json,"test_200k_24nov_swapped_train.json")
with open(output_json, "w") as f:
    json.dump(dict_fake_train, f, indent=4)

dict_fake_test = {k:dict_swap[k] for k in list(df_test['image_path'])}
output_json = os.path.join(dir_json,"test_200k_24nov_swapped_test.json")
with open(output_json, "w") as f:
    json.dump(dict_fake_test, f, indent=4)

In [ ]:
merged_dict_train = {**dict_real_train,**dict_fake_train}
output_json = os.path.join(dir_json,"test_200k_24nov_merged_train.json")
with open(output_json, "w") as f:
    json.dump(merged_dict_train, f, indent=4)

merged_dict_test = {**dict_real_test,**dict_fake_test}
output_json = os.path.join(dir_json,"test_200k_merged_test.json")
with open(output_json, "w") as f:
    json.dump(merged_dict_test, f, indent=4)

Collect all splits for all deepfake methods

In [111]:
len(dict_result)

NameError: name 'dict_result' is not defined

In [113]:
#for key in my_dict.keys():
#    if key.startswith(prefix):
#        matching_keys.append(key)

#data_source["/mnt/ssd/datasets/deepfake/200k_live_face_dataset/live/"+list_all[0]+".jpg.jpg"]

In [114]:
data_source[list(data_source.keys())[0]]

{'label': 0,
 'method': 'production_live',
 'processed_path': '/mnt/ssd/datasets/deepfake/200k_live_face_dataset/processes/live/face/170203_verify_8b73353a-ea41-4820-a855-0035c2b4a5c5_d84a7070-fe9a-4c23-826c-134650e4783a_original.jpg.jpg',
 'landmarks_path': '/mnt/ssd/datasets/deepfake/200k_live_face_dataset/processes/live/landmarks/170203_verify_8b73353a-ea41-4820-a855-0035c2b4a5c5_d84a7070-fe9a-4c23-826c-134650e4783a_original.jpg_landmarks.npy',
 'mask_path': '/mnt/ssd/datasets/deepfake/200k_live_face_dataset/processes/live/mask/170203_verify_8b73353a-ea41-4820-a855-0035c2b4a5c5_d84a7070-fe9a-4c23-826c-134650e4783a_original.jpg_mask.png',
 'angle': 0,
 'status': 'FACE DETECTED',
 'pred_age': '10-19',
 'score_blur_face': 64.37224850479575,
 'score_dark': 6.042599850363189,
 'score_blur_img': 30.31190234398195,
 'score_dfd1-0-0': 0.05273336172103882,
 'path': '/mnt/ssd/datasets/deepfake/200k_live_face_dataset/live/170203_verify_8b73353a-ea41-4820-a855-0035c2b4a5c5_d84a7070-fe9a-4c23-82

In [115]:
# Extract 'target' from file path
dict_att['0']['image_path'].split('_to_')[1]
# get 'target' image property

NameError: name 'dict_att' is not defined

## Notes

1. Gather info for all `swapped_id`, use `data_source` and `attributes` [ DONE ]
2. Stratify split by Target Info (`pred_gender`), Deepfake Method [DONE]
3. Re-collect the real files information from `source_name` and `target_name`. Output: `dict_real_train` [DONE]
4. Save to JSON
5. Optional: Combine all live and fake images into one dataset JSON

Expected outcome:
1. JSON of train real images
2. JSON of test real images
3. JSON of train fake images
4. JSON of test fake images